In [1]:
#libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
#data loading

data1 = pd.read_csv('train.csv')
data2 = pd.read_csv('test.csv')

train_df = data1.copy()
test_df = data2.copy()

print("First 5 rows of train_df: \n", train_df.head())

print("First 5 rows of test_df: \n", test_df.head())

First 5 rows of train_df: 
    id  annual_income  debt_to_income_ratio  credit_score  loan_amount  \
0   0       29367.99                 0.084           736      2528.42   
1   1       22108.02                 0.166           636      4593.10   
2   2       49566.20                 0.097           694     17005.15   
3   3       46858.25                 0.065           533      4682.48   
4   4       25496.70                 0.053           665     12184.43   

   interest_rate  gender marital_status education_level employment_status  \
0          13.67  Female         Single     High School     Self-employed   
1          12.92    Male        Married        Master's          Employed   
2           9.76    Male         Single     High School          Employed   
3          16.10  Female         Single     High School          Employed   
4          10.21    Male        Married     High School          Employed   

         loan_purpose grade_subgrade  loan_paid_back  
0              

In [3]:
train_df = train_df.drop(columns=['id'], axis=1)

test_id_placeholder = test_df['id']
test_df = test_df.drop(columns=['id'], axis=1)

In [4]:
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

In [5]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

X = train_df.drop(columns='loan_paid_back')
y = train_df['loan_paid_back']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
cat_model = CatBoostClassifier(cat_features=cat_cols)

cat_model.fit(X_train, y_train)

Learning rate set to 0.143205
0:	learn: 0.5405242	total: 411ms	remaining: 6m 50s
1:	learn: 0.4433729	total: 642ms	remaining: 5m 20s
2:	learn: 0.3802827	total: 912ms	remaining: 5m 3s
3:	learn: 0.3410668	total: 1.17s	remaining: 4m 50s
4:	learn: 0.3153787	total: 1.44s	remaining: 4m 46s
5:	learn: 0.2976656	total: 1.75s	remaining: 4m 50s
6:	learn: 0.2869880	total: 2.02s	remaining: 4m 46s
7:	learn: 0.2783549	total: 2.31s	remaining: 4m 45s
8:	learn: 0.2725982	total: 2.54s	remaining: 4m 40s
9:	learn: 0.2684360	total: 2.81s	remaining: 4m 37s
10:	learn: 0.2655688	total: 3.07s	remaining: 4m 36s
11:	learn: 0.2634373	total: 3.34s	remaining: 4m 34s
12:	learn: 0.2613712	total: 3.62s	remaining: 4m 34s
13:	learn: 0.2598790	total: 3.87s	remaining: 4m 32s
14:	learn: 0.2589231	total: 4.16s	remaining: 4m 33s
15:	learn: 0.2582219	total: 4.43s	remaining: 4m 32s
16:	learn: 0.2578017	total: 4.68s	remaining: 4m 30s
17:	learn: 0.2572070	total: 4.93s	remaining: 4m 29s
18:	learn: 0.2567331	total: 5.18s	remaining: 

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns: {cat_cols}")



Categorical columns: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']


In [17]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

catboost_pipeline = Pipeline([
    ('model', CatBoostClassifier(
        iterations=1000,
        cat_features=cat_cols,
        verbose=False,
        random_state=42
    ))
])

rf_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

xgb_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', XGBClassifier(n_estimators=100, random_state=42))
])

In [18]:
voting_ensemble = VotingClassifier(
    estimators=[
        ('catboost', catboost_pipeline),
        ('random_forest', rf_pipeline),
        ('xgboost', xgb_pipeline)
    ],
    voting='soft',
    weights=[0.4, 0.3, 0.3]
)

In [19]:
print("Training voting ensemble...")
voting_ensemble.fit(X_train, y_train)
print("Training complete!")

y_pred_proba = voting_ensemble.predict_proba(X_test)[:, 1]
y_pred = voting_ensemble.predict(X_test)

Training voting ensemble...


KeyboardInterrupt: 

In [ ]:
X_submission = test_df.copy()

test_pred = voting_ensemble.predict(test_df)

submission_df = pd.DataFrame({
    "id": test_id_placeholder,
    "loan_paid_back": test_pred
})

submission_df.to_csv('submission7.csv',index=False)
print("Success!")